## Checking employment trends in R

In [1]:
# Importing relevant datasets:
# (I installed all of these via Conda, since I was having trouble with
# alternative installation methods on Linux.)
# based on
# https://tidy-survey-r.github.io/tidy-survey-book/c05-descriptive-analysis.html
library(tidyverse)
library(broom)
library(survey)
library(srvyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: grid

Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loading required package: survival


Attaching package: ‘survey’


The following object is masked from ‘package:graphics’:

    dotchart



Attaching package: ‘srvyr’


The following object is masked from ‘package:stats’:

    filter




In [2]:
df <- read_csv('/home/kjb3/D1V1/kjb3largefiles/large_programming/datasets/cps/cps_employment_trends_revised.csv',
col_types = list(year_as_str = col_character()))


Checking the number of rows in the DataFrame:
(See https://dplyr.tidyverse.org/reference/n_distinct.html?q=nrow#null)

In [3]:
nrow(df)

[1] 23488604

In [4]:
# df %>% glimpse()

The following cell provides as somewhat 'hacky' way to specify 2026 as the baseline for an upcoming logistic regression. (It does so by making it the first year when its field is sorted in alphabetical order.)

(See https://dplyr.tidyverse.org/articles/recoding-replacing.html)

In [5]:
df <- df %>% mutate(
year_as_str_last_year_first = year_as_str %>% replace_when(
year_as_str == '2026' ~ '0_2026'))


Creating a survey design object:

(See https://tidy-survey-r.github.io/tidy-survey-book/c10-sample-designs-replicate-weights.html)

In [6]:
sd_df <- df %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)

Creating an object just for 2025 data in order to speed up processing times:

(Since the original CPS data was at the monthly level, I don't think that filtering out years at this stage would distort our results as would, say, filtering the dataset to include *only* software developers.) (See https://tidy-survey-r.github.io/tidy-survey-book/c04-getting-started.html#c04-getting-started for an explanation of why you shouldn't prematurely filter variables out of your sample for a given period.)

In [7]:
df_25 <- df %>% filter(year == 2025)
sd_df_25 <- df_25 %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)
# sd_df_25

In [8]:
nrow(df_25)

[1] 1041831

Creating an object just for 2022 through 2026 (which will prove useful for logistic regressions):

In [9]:
df_22_26 <- df %>% filter(year %in% c(2022, 2023, 2024, 2025, 2026))
sd_df_22_26 <- df_22_26 %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)
# sd_df_22_26

In [10]:
nrow(df_22_26)

[1] 5379704

## Data checks

### 1. Software-developer employment totals by year

In [11]:
sd_df_25_employed_devs_only <- sd_df_25 %>% filter(
occ2010 == 'Software developers, applications and systems software' & employed == 1)

The following code is based on https://tidy-survey-r.github.io/tidy-survey-book/c05-descriptive-analysis.html .

In [13]:
df_dev_emp_2025 = sd_df_25_employed_devs_only %>% survey_count(
year, vartype = c("se", "ci", "var", "cv"))
df_dev_emp_2025

year,n,n_se,n_low,n_upp,n_var,n_cv
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025,2331220,60146.87,2213263,2449176,3617646432,0.0258006


These standard-error, confidence-itnerval, and CV values match those that I got in svy. (See 'Employment totals by year' section in employment_data_analyses.ipynb.)

### 2. Logistic regression that evaluates whether the proportion of individuals ages 22-65 who are employed as developers has increased significantly from previous years to the current year:

In [14]:
sd_df_22_26_ages_22_65_only <- sd_df_22_26 %>% filter(age >= 22 & age <= 65)

In [18]:
sd_df_ages_22_65_only <- sd_df %>% filter(age >= 22 & age <= 65)

The following logistic-regression code is based on: https://tidy-survey-r.github.io/tidy-survey-book/c07-modeling.html

For a comparison of svy and R's logistic-regression approaches, see: https://svylab.com/learn/notes/posts/svy-vs-r-comparison/index.html

In [19]:
df_dev_emp_prop_by_year_reg_results_full <- sd_df_ages_22_65_only  %>% svyglm(
formula = employed_as_software_developer ~ year_as_str_last_year_first,
design = ., na.action = na.omit, family = quasibinomial)

df_dev_emp_prop_by_year_reg_results_full

Warning message in summary.glm(g):
“observations with zero weight not used for calculating dispersion”
Warning message in summary.glm(glm.object):
“observations with zero weight not used for calculating dispersion”


Stratified 1 - level Cluster Sampling design (with replacement)
With (1400609) clusters.
Called via srvyr
Sampling variables:
  - ids: cpsid 
  - strata: final_stratum_revision 
  - weights: year_wtfinl 

Call:  svyglm(formula = employed_as_software_developer ~ year_as_str_last_year_first, 
    design = ., family = quasibinomial, na.action = na.omit)

Coefficients:
                    (Intercept)  year_as_str_last_year_first2010  
                        -4.3831                          -0.8682  
year_as_str_last_year_first2011  year_as_str_last_year_first2012  
                        -0.8503                          -0.8267  
year_as_str_last_year_first2013  year_as_str_last_year_first2014  
                        -0.8209                          -0.7159  
year_as_str_last_year_first2015  year_as_str_last_year_first2016  
                        -0.6231                          -0.5441  
year_as_str_last_year_first2017  year_as_str_last_year_first2018  
                        -0.50

Not sure why the following tidy() calls are taking so long:

In [20]:
tidy(df_dev_emp_prop_by_year_reg_results_full)

term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-4.3831093,0.03082686,-142.184744,0.000000e+00
year_as_str_last_year_first2010,-0.8681984,0.04348948,-19.963413,1.179244e-88
year_as_str_last_year_first2011,-0.8503177,0.04292528,-19.809254,2.546677e-87
year_as_str_last_year_first2012,-0.8266764,0.04265729,-19.379485,1.179417e-83
year_as_str_last_year_first2013,-0.8209008,0.04223490,-19.436552,3.886126e-84
year_as_str_last_year_first2014,-0.7158733,0.04215228,-16.983025,1.113371e-64
year_as_str_last_year_first2015,-0.6231320,0.04166683,-14.955110,1.455484e-50
year_as_str_last_year_first2016,-0.5440654,0.04088611,-13.306851,2.123954e-40
year_as_str_last_year_first2017,-0.5069612,0.04069718,-12.456911,1.288022e-35


The p-values within this output are actually a bit smaller than those within my svy-based Python analysis, even though I *believe* I have the same setup for each function (except that my Python dataset includes more years--which *shouldn't* make a difference here). I might reach out to svy's developer to get his thoughts on what might be going on here. (It's quite possible that the `survey` and `svy` libraries use different default settings for these sorts of regresisons, or that I need to specify the same tolerance setting for both functions; see https://svylab.com/learn/notes/posts/svy-vs-r-comparison/#generalized-linear-models-glms)

### 3. Logistic regression that evaluates whether software employment *by age group* has increased significantly from previous years to the most recent year

(Coming soon!)